In [2]:
import pdfplumber
import pandas as pd
import re
import os

# -------------------------------
# Grade Mapping
# -------------------------------
GRADE_POINTS = {
    "A+": 10, "A": 9, "B+": 8, "B": 7,
    "C+": 6, "C": 5, "D": 4,
    "E": 0, "F": 0, "I": 0
}

# ✅ FIX: Proper regex order (longer first)
GRADE_REGEX = r'(A\+|B\+|C\+|A|B|C|D|E|F|I)'

# -------------------------------
# Extract Student Info
# -------------------------------
def extract_student_info(text):
    uid = re.search(r'UID\s*([A-Z0-9]+)', text)
    name = re.search(r'Name\s*(.+)', text)

    return (
        uid.group(1) if uid else None,
        name.group(1).strip() if name else None
    )

# -------------------------------
# Extract Subjects
# -------------------------------
def extract_subjects(text):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    subjects = []
    i = 0

    while i < len(lines):
        line = lines[i]

        # -------------------------
        # CASE 1: Full single-line subject
        # -------------------------
        full = re.search(
            rf'([A-Z0-9\-]+)\s+(.+?)\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)\s+{GRADE_REGEX}',
            line
        )
        if full:
            subjects.append([
                full.group(1), full.group(2),
                full.group(3), full.group(4),
                full.group(5), full.group(6)
            ])
            i += 1
            continue

        # -------------------------
        # CASE 2: 3-line subject (incl. General Proficiency)
        # -------------------------
        if re.match(r'^\d{2}[A-Z]{3,4}-$', line):
            try:
                line2 = lines[i+1]
                line3 = lines[i+2]

                subject_code = line + line3

                match_full = re.search(
                    rf'(.+?)\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)\s+{GRADE_REGEX}',
                    line2
                )

                match_simple = re.search(
                    rf'(.+?)\s+([\d\.]+)\s+([\d\.]+)\s+{GRADE_REGEX}',
                    line2
                )

                if match_full:
                    subjects.append([
                        subject_code,
                        match_full.group(1),
                        match_full.group(2),
                        match_full.group(3),
                        match_full.group(4),
                        match_full.group(5)
                    ])
                elif match_simple:
                    subjects.append([
                        subject_code,
                        match_simple.group(1),
                        None,
                        match_simple.group(2),
                        match_simple.group(3),
                        match_simple.group(4)
                    ])

                i += 3
                continue
            except:
                pass

        # -------------------------
        # CASE 3: Social Internship
        # -------------------------
        if "Internship" in line:
            try:
                combined = line + " " + lines[i+1] + " " + lines[i+2]

                match = re.search(
                    rf'([A-Z0-9\-]+)\s+([\d\.]+)\s+{GRADE_REGEX}',
                    combined
                )

                if match:
                    subjects.append([
                        match.group(1),
                        "Social Internship",
                        None,
                        None,
                        match.group(2),
                        match.group(3)
                    ])

                i += 3
                continue
            except:
                pass

        i += 1

    return pd.DataFrame(subjects, columns=[
        "Subject Code", "Subject Name",
        "Internal", "External",
        "Credits", "Grade"
    ])

# -------------------------------
# Clean Data
# -------------------------------
def clean_data(df, uid, name):
    for col in ["Internal", "External", "Credits"]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # ✅ FIX: Normalize grade formatting
    df["Grade"] = df["Grade"].astype(str).str.replace(" ", "").str.strip()

    df["UID"] = uid
    df["Name"] = name

    return df

# -------------------------------
# Calculate CGPA
# -------------------------------
def calculate_cgpa(df):
    df["Grade Points"] = df["Grade"].map(GRADE_POINTS)

    # Keep only valid rows
    valid_df = df[
        (df["Credits"].notna()) &
        (df["Credits"] > 0) &
        (df["Grade Points"].notna())
    ].copy()

    valid_df["Credit Points"] = valid_df["Credits"] * valid_df["Grade Points"]

    total_credit_points = valid_df["Credit Points"].sum()
    total_credits = valid_df["Credits"].sum()

    if total_credits == 0:
        return 0

    return round(total_credit_points / total_credits, 2)

# -------------------------------
# MAIN FUNCTION (FOLDER BASED)
# -------------------------------
def process_multiple_pdfs(folder_path, output_csv="all_students.csv"):
    all_students = []

    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]

    if not pdf_files:
        print("❌ No PDF files found in folder")
        return

    for file in pdf_files:
        pdf_path = os.path.join(folder_path, file)

        print(f"\n📄 Processing: {file}")

        try:
            with pdfplumber.open(pdf_path) as pdf:
                text = pdf.pages[0].extract_text()

            uid, name = extract_student_info(text)
            df = extract_subjects(text)

            if df.empty:
                print("⚠️ Skipped (no data extracted)")
                continue

            df = clean_data(df, uid, name)

            cgpa = calculate_cgpa(df)
            df["CGPA"] = cgpa

            all_students.append(df)

            print(f"✅ {name} | Subjects: {len(df)} | CGPA: {cgpa}")

        except Exception as e:
            print(f"❌ Error processing {file}: {e}")

    final_df = pd.concat(all_students, ignore_index=True)

    # Save CSV safely
    try:
        final_df.to_csv(output_csv, index=False)
    except PermissionError:
        new_file = "all_students_new.csv"
        final_df.to_csv(new_file, index=False)
        print(f"⚠️ File in use. Saved as {new_file}")
        return final_df

    print("\n🎉 FINAL DATASET CREATED")
    print("Total Students:", final_df["UID"].nunique())

    return final_df

# -------------------------------
# RUN
# -------------------------------
if __name__ == "__main__":
    process_multiple_pdfs("results")


📄 Processing: Arushi.pdf
✅ ARUSHI GARG | Subjects: 9 | CGPA: 8.83

📄 Processing: Mahi.pdf
⚠️ Skipped (no data extracted)

📄 Processing: Prabhjot.pdf
✅ PRABHJOT SINGH | Subjects: 9 | CGPA: 7.96

📄 Processing: Ridam.pdf
✅ RIDAM | Subjects: 9 | CGPA: 8.52

📄 Processing: Tarun.pdf
✅ TARUN GARG | Subjects: 9 | CGPA: 9.04

🎉 FINAL DATASET CREATED
Total Students: 4
